## Sección 4: Data Cleaning (Limpieza de Datos) - Guía Definitiva
El objetivo de la limpieza es transformar datos "sucios" (inconsistentes, incompletos, erróneos) en un formato limpio y robusto para el análisis. La regla de oro: Antes de imputar o eliminar, ENTENDER el contexto del negocio/dato.

### 4.1 Diagnóstico Inicial de Calidad (El Check-up)
Antes de tocar nada, hay que saber qué está roto.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# --- DATAFRAME DE EJEMPLO CON TODOS LOS PROBLEMAS POSIBLES ---
datos_sucios = {
    'Nombre': ['Ana', 'Luis', 'Carlos', 'Marta', 'Pedro', None, '  Sofia  '],
    'Edad': [23, 35, None, 41, -5, 29, 100],  # Edad negativa y 100 (posible error)
    'Salario': ['$50,000', '$75,000', None, '$120,000', '$45,000', '$60,000', 'Desconocido'],
    'Fecha_Ingreso': ['2021-01-15', '20/02/2020', '2021-03-01', '2020-04-10', '2022-05-20', '2021-06-15', '2021-07-01'],
    'Ciudad': ['Madrid', 'Madrid', 'Valencia', 'Sevilla', 'Barcelona', 'Madrid', 'Madrid'],
    'Activo': [True, True, False, True, None, True, False]
}
df = pd.DataFrame(datos_sucios)
print("--- DATAFRAME ORIGINAL ---")
print(df, '\n')

# 1. Resumen general (tipos, nulos y memoria)
print("\n--- INFO() ---")
df.info()

# 2. Porcentaje de nulos por columna (ORDENADO de mayor a menor)
print("\n--- PORCENTAJE DE NULOS ---")
print((df.isnull().sum() / len(df)) * 100, '\n')

# 3. Cardinalidad (valores únicos) - útil para categóricas
print("\n--- VALORES ÚNICOS POR COLUMNA ---", '\n')
for col in df.columns:
    print(f"{col}: {df[col].nunique()} únicos")

# 4. Estadísticas básicas (detectar mins/maxs extraños)
print("\n--- DESCRIBE (Numéricas) ---")
print(df.describe(), '\n')  # Mínimo -5 y máximo 100 saltan a la vista

# 5. Vista de los valores atípicos en categóricas/texto
print("\n--- TOP 5 FRECUENCIAS (Ciudad) ---")
print(df['Ciudad'].value_counts(), '\n')

--- DATAFRAME ORIGINAL ---
      Nombre   Edad      Salario Fecha_Ingreso     Ciudad Activo
0        Ana   23.0      $50,000    2021-01-15     Madrid   True
1       Luis   35.0      $75,000    20/02/2020     Madrid   True
2     Carlos    NaN          NaN    2021-03-01   Valencia  False
3      Marta   41.0     $120,000    2020-04-10    Sevilla   True
4      Pedro   -5.0      $45,000    2022-05-20  Barcelona   None
5        NaN   29.0      $60,000    2021-06-15     Madrid   True
6    Sofia    100.0  Desconocido    2021-07-01     Madrid  False 


--- INFO() ---
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Nombre         6 non-null      str    
 1   Edad           6 non-null      float64
 2   Salario        6 non-null      str    
 3   Fecha_Ingreso  7 non-null      str    
 4   Ciudad         7 non-null      str    
 5   Activo         6 non-null      ob

### 4.2 Manejo de Duplicados
Los duplicados distorsionan métricas como medias y asignan pesos incorrectos a ciertas observaciones.

In [3]:
# Detectar duplicados (misma fila exacta)
duplicados = df.duplicated()
print(f"¿Filas duplicadas exactas?: {duplicados.sum()}", '\n')

# Para este ejemplo, duplicamos una fila a propósito
df_con_dup = pd.concat([df, df.iloc[[0]]], ignore_index=True)
print(f"Tamaño con duplicados: {len(df_con_dup)}", '\n')

# --- ESTRATEGIAS ---

# 1. Eliminar duplicados exactos (todas las columnas)
df_limpio_dup = df_con_dup.drop_duplicates()
print(f"Tamaño tras eliminar duplicados exactos: {len(df_limpio_dup)}", '\n')

# 2. Eliminar duplicados basados en un subset (ej: si el Nombre y Ciudad se repiten)
df_sin_dup_subset = df_con_dup.drop_duplicates(subset=['Nombre', 'Ciudad'], keep='first')
# keep='first' (default) conserva la primera, keep='last' conserva la última, keep=False borra todas

# 3. Identificar qué filas son duplicadas de otra
df['es_duplicado'] = df.duplicated(keep=False)  # Marca todos los duplicados (incluido el primero)
print(df[df['es_duplicado'] == True], '\n')

¿Filas duplicadas exactas?: 0 

Tamaño con duplicados: 8 

Tamaño tras eliminar duplicados exactos: 7 

Empty DataFrame
Columns: [Nombre, Edad, Salario, Fecha_Ingreso, Ciudad, Activo, es_duplicado]
Index: [] 



### 4.3 Manejo de Datos Faltantes (NaN) - Estrategias de Eliminación
No siempre hay que eliminar. La decisión depende del % de nulos y la importancia de la columna.

In [4]:
# --- DIAGNÓSTICO AVANZADO DE NULOS ---

# Crear un DataFrame con más nulos para ejemplos
df_nulos = df.copy()
df_nulos.loc[0, 'Salario'] = None
df_nulos.loc[2, 'Ciudad'] = None
df_nulos.loc[5, 'Fecha_Ingreso'] = None

print("--- MATRIZ DE NULOS (Visual) ---")
print(df_nulos.isnull(), '\n')

# --- ESTRATEGIA 1: ELIMINAR FILAS (dropna) ---

# 1. Eliminar filas que tengan CUALQUIER NaN
df_drop_any = df_nulos.dropna(axis=0, how='any')
print(f"Eliminar any -> {len(df_nulos)} -> {len(df_drop_any)} filas", '\n')

# 2. Eliminar filas que tengan TODOS los valores NaN (rara vez útil)
df_drop_all = df_nulos.dropna(axis=0, how='all')

# 3. ELIMINAR POR UMBRAL (THRESH) - ¡MUY ÚTIL!
# Conserva filas que tengan al menos 'thresh' valores NO nulos.
# Si tengo 6 columnas, y pongo thresh=5, solo borro filas con 2 o más nulos.
df_thresh = df_nulos.dropna(axis=0, thresh=5)  # Conserva si tiene 5 o más datos
print(f"Eliminar con thresh=5 -> {len(df_nulos)} -> {len(df_thresh)} filas", '\n')

# 4. Eliminar filas donde una columna específica está nula (ej: columna objetivo)
df_drop_salario_nulo = df_nulos.dropna(subset=['Salario'])
print(f"Eliminar si Salario es nulo -> {len(df_nulos)} -> {len(df_drop_salario_nulo)}", '\n')

# --- ESTRATEGIA 2: ELIMINAR COLUMNAS ---
# Si una columna tiene > 60-70% de nulos, probablemente debas borrarla.
df_column_drop = df_nulos.drop(columns=['Salario'])  # Borrar columna entera
# O usando dropna en columnas
df_col_thresh = df_nulos.dropna(axis=1, thresh=int(len(df_nulos)*0.5))  # columnas con al menos 50% de no-nulos

--- MATRIZ DE NULOS (Visual) ---
   Nombre   Edad  Salario  Fecha_Ingreso  Ciudad  Activo  es_duplicado
0   False  False     True          False   False   False         False
1   False  False    False          False   False   False         False
2   False   True     True          False    True   False         False
3   False  False    False          False   False   False         False
4   False  False    False          False   False    True         False
5    True  False    False           True   False   False         False
6   False  False    False          False   False   False         False 

Eliminar any -> 7 -> 3 filas 

Eliminar con thresh=5 -> 7 -> 6 filas 

Eliminar si Salario es nulo -> 7 -> 5 



### 4.4 Manejo de Datos Faltantes (NaN) - Estrategias de Imputación
Imputar es RELLENAR. Cuidado: Imputar sin más puede introducir sesgo. Siempre analiza si el dato falta al azar (MCAR), al azar condicionado (MAR) o no al azar (MNAR).

In [5]:
# Creamos una copia limpia de números para imputar
df_imp = df[['Edad', 'Nombre', 'Ciudad']].copy()
df_imp.loc[2, 'Edad'] = np.nan
df_imp.loc[4, 'Edad'] = np.nan
df_imp.loc[0, 'Ciudad'] = np.nan

print("--- DATOS PARA IMPUTAR ---")
print(df_imp)

# --- IMPUTACIÓN PARA NUMÉRICAS ---

# 1. Imputar con una constante (ej: 0)
df_const = df_imp.fillna(0)
print("\nImputar con 0:\n", df_const)

# 2. Imputar con la MEDIA (si datos normales, sin outliers)
media_edad = df_imp['Edad'].mean()
df_media = df_imp.fillna({'Edad': media_edad})
print(f"\nImputar con Media ({media_edad:.2f}):\n", df_media)

# 3. Imputar con la MEDIANA (ROBUSTO ante outliers - RECOMENDADO por defecto)
mediana_edad = df_imp['Edad'].median()
df_mediana = df_imp.fillna({'Edad': mediana_edad})
print(f"\nImputar con Mediana ({mediana_edad}):\n", df_mediana)

# 4. Imputar con la MODA (para categóricas o numéricas discretas)
moda_ciudad = df_imp['Ciudad'].mode()[0]  # mode() devuelve una Serie
df_moda = df_imp.fillna({'Ciudad': moda_ciudad})
print(f"\nImputar Categoría con Moda ('{moda_ciudad}'):\n", df_moda)

# --- IMPUTACIÓN AVANZADA (POR GRUPOS) ---
# Ej: La edad media puede variar por ciudad. Esto es mucho mejor.
df_grupo = df[['Edad', 'Ciudad']].copy()
df_grupo.loc[2, 'Edad'] = np.nan
df_grupo.loc[4, 'Edad'] = np.nan

# Calculamos la media por grupo y la aplicamos con transform
medias_por_ciudad = df_grupo.groupby('Ciudad')['Edad'].transform('mean')
df_grupo['Edad_imputada_grupo'] = df_grupo['Edad'].fillna(medias_por_ciudad)
print("\n--- IMPUTACIÓN POR GRUPO (Media por Ciudad) ---")
print(df_grupo)

# --- IMPUTACIÓN CON SKLEARN (SimpleImputer) - Estándar en producción ---
from sklearn.impute import SimpleImputer

# Para numéricas
imp_mean = SimpleImputer(strategy='median')  # 'mean', 'median', 'constant'
df_num = df[['Edad']].copy()
df_num_imputado = pd.DataFrame(imp_mean.fit_transform(df_num), columns=['Edad'])
print("\nImputación con SimpleImputer (mediana):\n", df_num_imputado)

# Para categóricas (con 'most_frequent')
imp_cat = SimpleImputer(strategy='most_frequent')
df_cat = df[['Ciudad']].copy()
df_cat_imputado = pd.DataFrame(imp_cat.fit_transform(df_cat), columns=['Ciudad'])
print(df_cat_imputado)

# --- IMPUTACIÓN POR INTERPOLACIÓN (Para series temporales o secuenciales) ---
df_temp = pd.DataFrame({'Dia': [1, 2, 3, 4, 5], 'Ventas': [100, np.nan, 300, np.nan, 500]})
df_temp['Ventas_interpolada'] = df_temp['Ventas'].interpolate(method='linear')
print("\nInterpolación lineal:\n", df_temp)

--- DATOS PARA IMPUTAR ---
    Edad     Nombre     Ciudad
0   23.0        Ana        NaN
1   35.0       Luis     Madrid
2    NaN     Carlos   Valencia
3   41.0      Marta    Sevilla
4    NaN      Pedro  Barcelona
5   29.0        NaN     Madrid
6  100.0    Sofia       Madrid

Imputar con 0:
     Edad     Nombre     Ciudad
0   23.0        Ana          0
1   35.0       Luis     Madrid
2    0.0     Carlos   Valencia
3   41.0      Marta    Sevilla
4    0.0      Pedro  Barcelona
5   29.0          0     Madrid
6  100.0    Sofia       Madrid

Imputar con Media (45.60):
     Edad     Nombre     Ciudad
0   23.0        Ana        NaN
1   35.0       Luis     Madrid
2   45.6     Carlos   Valencia
3   41.0      Marta    Sevilla
4   45.6      Pedro  Barcelona
5   29.0        NaN     Madrid
6  100.0    Sofia       Madrid

Imputar con Mediana (35.0):
     Edad     Nombre     Ciudad
0   23.0        Ana        NaN
1   35.0       Luis     Madrid
2   35.0     Carlos   Valencia
3   41.0      Marta    Sevill

### 4.5 Manejo de Outliers (Valores Atípicos)
Los outliers pueden ser errores (ej: edad -5 o 200) o descubrimientos interesantes (ej: fraude). Nunca los elimines sin investigar.

In [6]:
# Reconstruimos la columna Edad con valores extremos
df_out = pd.DataFrame({'Edad': [23, 35, 29, 41, 12, 85, 29, 32, 19, 150, -10]})
print("--- DATOS CON OUTLIERS ---")
print(df_out)

# --- MÉTODO 1: RANGO INTERCUARTÍLICO (IQR) - EL MÁS ROBUSTO ---
Q1 = df_out['Edad'].quantile(0.25)
Q3 = df_out['Edad'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Límites: [{limite_inferior}, {limite_superior}]")

outliers_iqr = df_out[(df_out['Edad'] < limite_inferior) | (df_out['Edad'] > limite_superior)]
print(f"Outliers detectados por IQR:\n{outliers_iqr}")

# --- ESTRATEGIAS DE ACTUACIÓN ---

# 1. ELIMINAR los outliers (si son errores claros, ej: -10)
df_sin_outliers = df_out[(df_out['Edad'] >= limite_inferior) & (df_out['Edad'] <= limite_superior)]
print(f"\nTras eliminar outliers: {len(df_sin_outliers)} filas")

# 2. CAPEAR (WINSORIZING) - Llevar el valor al límite más cercano
# Esto es mejor que eliminar porque conservas la fila.
df_capeado = df_out.copy()
df_capeado['Edad'] = df_capeado['Edad'].clip(lower=limite_inferior, upper=limite_superior)
print(f"\nValores capeados al límite:\n{df_capeado}")

# 3. TRANSFORMACIÓN (LOG) - Para reducir el impacto de outliers en colas derechas
# Si los datos son positivos y tienen cola a la derecha (ingresos, precios)
df_out['Edad_log'] = np.log1p(df_out['Edad'])  # log(1+x) para evitar log(0)
print("\nTransformación log (reduce el impacto de 150):\n", df_out)

# 4. MÉTODO Z-SCORE (para distribuciones normales)
from scipy import stats
z_scores = np.abs(stats.zscore(df_out['Edad']))
outliers_z = df_out[z_scores > 3]
print(f"\nOutliers por Z-Score (>3):\n{outliers_z}")

--- DATOS CON OUTLIERS ---
    Edad
0     23
1     35
2     29
3     41
4     12
5     85
6     29
7     32
8     19
9    150
10   -10
Q1: 21.0, Q3: 38.0, IQR: 17.0
Límites: [-4.5, 63.5]
Outliers detectados por IQR:
    Edad
5     85
9    150
10   -10

Tras eliminar outliers: 8 filas

Valores capeados al límite:
    Edad
0   23.0
1   35.0
2   29.0
3   41.0
4   12.0
5   63.5
6   29.0
7   32.0
8   19.0
9   63.5
10  -4.5

Transformación log (reduce el impacto de 150):
     Edad  Edad_log
0     23  3.178054
1     35  3.583519
2     29  3.401197
3     41  3.737670
4     12  2.564949
5     85  4.454347
6     29  3.401197
7     32  3.496508
8     19  2.995732
9    150  5.017280
10   -10       NaN

Outliers por Z-Score (>3):
Empty DataFrame
Columns: [Edad, Edad_log]
Index: []


/home/ldan/miniconda3/envs/mineria/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


### 4.6 Limpieza de Strings (Texto y Categóricas)
Los textos vienen con espacios, mayúsculas/minúsculas, símbolos, etc. Estandarizar es clave.

In [7]:
df_str = pd.DataFrame({
    'Producto': ['  Laptop  ', 'Monitor', '  Mouse', 'Teclado ', 'LAPTOP'],
    'Precio_texto': ['$1,200', '$800', 'No disponible', '$150', '$2,500']
})
print("--- STRINGS SUCOS ---")
print(df_str)

# 1. Eliminar espacios en blanco al inicio/final (strip)
df_str['Producto'] = df_str['Producto'].str.strip()
print("\nTras strip:\n", df_str['Producto'])

# 2. Estandarizar mayúsculas/minúsculas (para agrupar correctamente)
df_str['Producto'] = df_str['Producto'].str.lower()  # o .upper() .title()
print("\nTras lower():\n", df_str['Producto'])

# 3. Reemplazar caracteres o patrones (limpiar símbolos)
# Quitar '$' y ',' del precio
df_str['Precio_limpio'] = df_str['Precio_texto'].str.replace('$', '', regex=False)
df_str['Precio_limpio'] = df_str['Precio_limpio'].str.replace(',', '', regex=False)
print("\nTras quitar $ y ,:\n", df_str['Precio_limpio'])

# 4. Extraer números de un texto con Regex (cuando hay texto mezclado)
df_str['Precio_num'] = df_str['Precio_texto'].str.extract(r'(\d+\.?\d*)').astype(float)
print("\nExtracción con regex (solo números):\n", df_str['Precio_num'])

# 5. Detectar si contiene cierta palabra (para hacer banderas)
df_str['es_Laptop'] = df_str['Producto'].str.contains('laptop', case=False, na=False)
print("\nContiene 'laptop' (case insensitive):\n", df_str['es_Laptop'])

# 6. Dividir columnas por un delimitador
df_split = pd.DataFrame({'Nombre_Completo': ['Ana García', 'Luis Pérez', 'Carlos Ruiz']})
df_split[['Nombre', 'Apellido']] = df_split['Nombre_Completo'].str.split(' ', expand=True)
print("\nSplit de nombres:\n", df_split)

# 7. Rellenar categóricas poco frecuentes con 'Otros' (para evitar overfitting)
frecuencias = df['Ciudad'].value_counts()
categorias_raras = frecuencias[frecuencias < 2].index  # Ciudades con menos de 2 apariciones
df['Ciudad_agrupada'] = df['Ciudad'].replace(categorias_raras, 'Otros')
print("\nCategorías raras agrupadas en 'Otros':\n", df[['Ciudad', 'Ciudad_agrupada']])

--- STRINGS SUCOS ---
     Producto   Precio_texto
0    Laptop           $1,200
1     Monitor           $800
2       Mouse  No disponible
3    Teclado            $150
4      LAPTOP         $2,500

Tras strip:
 0     Laptop
1    Monitor
2      Mouse
3    Teclado
4     LAPTOP
Name: Producto, dtype: str

Tras lower():
 0     laptop
1    monitor
2      mouse
3    teclado
4     laptop
Name: Producto, dtype: str

Tras quitar $ y ,:
 0             1200
1              800
2    No disponible
3              150
4             2500
Name: Precio_limpio, dtype: str

Extracción con regex (solo números):
 0      1.0
1    800.0
2      NaN
3    150.0
4      2.0
Name: Precio_num, dtype: float64

Contiene 'laptop' (case insensitive):
 0     True
1    False
2    False
3    False
4     True
Name: es_Laptop, dtype: bool

Split de nombres:
   Nombre_Completo  Nombre Apellido
0      Ana García     Ana   García
1      Luis Pérez    Luis    Pérez
2     Carlos Ruiz  Carlos     Ruiz

Categorías raras agrupadas en 

### 4.7 Limpieza de Fechas (Datetime)
Las fechas vienen en formatos locos. Hay que unificarlas y extraer información útil.

In [8]:
df_fechas = pd.DataFrame({
    'Fecha': ['2021-01-15', '20/02/2020', '2021-03-01', '2020-04-10', '2022-05-20', '2021/06/15', '01-07-2021']
})
print("--- FECHAS SUCAS ---")
print(df_fechas)

# 1. Convertir a datetime (con errors='coerce' para que no explote)
# dayfirst=True si el día va primero (formato europeo)
df_fechas['Fecha_parseada'] = pd.to_datetime(df_fechas['Fecha'], errors='coerce', dayfirst=False)
print("\nParseo automático (coerce pone NaT si no puede):\n", df_fechas)

# Si falla, probar con formatos específicos (MUCHO MÁS RÁPIDO)
# df_fechas['Fecha_parseada'] = pd.to_datetime(df_fechas['Fecha'], format='%d/%m/%Y', errors='coerce')

# 2. Extraer componentes (año, mes, día, día de la semana)
df_fechas['Año'] = df_fechas['Fecha_parseada'].dt.year
df_fechas['Mes'] = df_fechas['Fecha_parseada'].dt.month
df_fechas['Dia'] = df_fechas['Fecha_parseada'].dt.day
df_fechas['Dia_Semana'] = df_fechas['Fecha_parseada'].dt.dayofweek  # 0=Lunes, 6=Domingo
df_fechas['Nombre_Dia'] = df_fechas['Fecha_parseada'].dt.day_name()
print("\nComponentes extraídos:\n", df_fechas)

# 3. Calcular diferencias entre fechas (antigüedad)
df_fechas['Fecha_actual'] = pd.Timestamp.now()
df_fechas['Antigüedad_dias'] = (df_fechas['Fecha_actual'] - df_fechas['Fecha_parseada']).dt.days
print("\nAntigüedad en días:\n", df_fechas[['Fecha_parseada', 'Antigüedad_dias']])

--- FECHAS SUCAS ---
        Fecha
0  2021-01-15
1  20/02/2020
2  2021-03-01
3  2020-04-10
4  2022-05-20
5  2021/06/15
6  01-07-2021

Parseo automático (coerce pone NaT si no puede):
         Fecha Fecha_parseada
0  2021-01-15     2021-01-15
1  20/02/2020            NaT
2  2021-03-01     2021-03-01
3  2020-04-10     2020-04-10
4  2022-05-20     2022-05-20
5  2021/06/15            NaT
6  01-07-2021            NaT

Componentes extraídos:
         Fecha Fecha_parseada     Año  Mes   Dia  Dia_Semana Nombre_Dia
0  2021-01-15     2021-01-15  2021.0  1.0  15.0         4.0     Friday
1  20/02/2020            NaT     NaN  NaN   NaN         NaN        NaN
2  2021-03-01     2021-03-01  2021.0  3.0   1.0         0.0     Monday
3  2020-04-10     2020-04-10  2020.0  4.0  10.0         4.0     Friday
4  2022-05-20     2022-05-20  2022.0  5.0  20.0         4.0     Friday
5  2021/06/15            NaT     NaN  NaN   NaN         NaN        NaN
6  01-07-2021            NaT     NaN  NaN   NaN         NaN   

### 4.8 Limpieza de Tipos de Datos
A veces, números están guardados como texto o booleanos como objetos.

In [9]:
df_tipos = pd.DataFrame({
    'ID': ['1', '2', '3', '4'],
    'Precio': ['100', '200', 'trescientos', '400'],  # 'trescientos' es un error
    'Activo': ['Sí', 'No', 'Sí', 'No']
})
print("--- TIPOS DE DATOS ORIGINALES ---")
print(df_tipos.dtypes)

# 1. Convertir a numérico (errors='coerce' convierte lo no convertible a NaN)
df_tipos['Precio_num'] = pd.to_numeric(df_tipos['Precio'], errors='coerce')
print("\nPrecio convertido a numérico (coerce):\n", df_tipos[['Precio', 'Precio_num']])

# 2. Convertir a categórico (ahorra memoria y mejora rendimiento)
df_tipos['ID_cat'] = df_tipos['ID'].astype('category')
print(df_tipos['ID_cat'].dtype)

# 3. Mapear booleanos desde texto
mapa_si_no = {'Sí': 1, 'No': 0}
df_tipos['Activo_bin'] = df_tipos['Activo'].map(mapa_si_no)
print("\nMapeo de Sí/No a 1/0:\n", df_tipos[['Activo', 'Activo_bin']])

# 4. Convertir múltiples columnas a la vez
columnas_float = ['Precio_num']
df_tipos[columnas_float] = df_tipos[columnas_float].astype(float)

--- TIPOS DE DATOS ORIGINALES ---
ID        str
Precio    str
Activo    str
dtype: object

Precio convertido a numérico (coerce):
         Precio  Precio_num
0          100       100.0
1          200       200.0
2  trescientos         NaN
3          400       400.0
category

Mapeo de Sí/No a 1/0:
   Activo  Activo_bin
0     Sí           1
1     No           0
2     Sí           1
3     No           0



---

## 4.9 Casos Prácticos Reales (Situaciones Comunes)

**Caso 1:** Columna con texto y números (ej: "100€", "USD 50").
**Solución:** Usar `str.extract` con regex y luego `pd.to_numeric`.

**Caso 2:** Faltan datos en una columna numérica muy sesgada (ej: Ingresos).
**Solución:** Imputar con la **mediana** o, mejor, con la **mediana del grupo** (ej: por ciudad o categoría). Si el sesgo es brutal, imputar con 0 solo tiene sentido si el 0 es un valor real (ej: ventas).

**Caso 3:** Faltan datos no al azar (MNAR). Ej: personas con altos ingresos tienden a no reportar su salario.
**Solución:** NO imputes a la ligera. Crea una variable bandera (`df['salario_faltante'] = df['salario'].isnull()`) para que el modelo aprenda que el hecho de que falte es una señal en sí misma. Luego imputa con un valor extremo o la mediana.

**Caso 4:** Outliers en variables de distribución exponencial (ej: tiempo de espera).
**Solución:** Aplicar transformación **logarítmica** (`np.log1p`) para reducir el impacto, en lugar de eliminarlos.

**Caso 5:** Fechas en formato americano vs europeo (MM/DD vs DD/MM).
**Solución:** Usar `dayfirst=True` en `pd.to_datetime` o especificar el `format` explícitamente para evitar confusiones (ej: `format='%d/%m/%Y'`).

**Caso 6:** Duplicados con ligeras variantes (ej: "Madrid" vs "MADRID" vs "Madrid ").
**Solución:** Aplicar `str.strip().str.lower()` ANTES de detectar duplicados.

---